## 2. VECM

In [ ]:
import pandas as pd
from pathlib import Path

### 2.1 Consolidated Results

In [ ]:
def load_all_csvs(directory_path):
    data_dir = Path(directory_path)
    csv_files = list(data_dir.glob("*.csv"))

    if not csv_files:
        return None

    all_dataframes = []

    for file in csv_files:
        try:
            df = pd.read_csv(file)
            all_dataframes.append(df)
        except Exception as e:
            print(f"Error loading {file.name}: {e}")

    if all_dataframes:
        return pd.concat(all_dataframes, ignore_index=True)
    return None

file_dir = "../../results/02_vector_error_correction_model"
VECM_master = load_all_csvs(file_dir)

In [ ]:
# Bringing the "event_date" into a standardized format
VECM_master["event_date"] = pd.to_datetime(VECM_master["event_date"], format = "mixed")
VECM_master["event_date"] = VECM_master["event_date"].dt.strftime("%Y-%m-%d")
VECM_master['relationship'] = VECM_master['source_market'] + ' -> ' + VECM_master['target_market']

In [3]:
VECM_master = VECM_master.rename(columns={"is_significant":"is_sig_5pct"})

In [4]:
# Function to calculate conditional stats (only for significant tests)
def get_conditional_stats(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats
    grouped = significant_data.groupby('relationship')[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# Process 5% Significance
df_5pct = VECM_master.groupby('relationship')['is_sig_5pct'].agg(['sum', 'count', 'mean'])
df_5pct = df_5pct.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# Get avg coefficient and std for 5% significant results
stats_5pct = get_conditional_stats(VECM_master, 'is_sig_5pct', 'VECM_coefficient')
df_5pct = df_5pct.join(stats_5pct)

# Sort and display
df_5pct = df_5pct.sort_values(by='significance_rate_5pct', ascending=False)

# Ordering the data in the desired manner before printing
column_order_5pct = [
    'total_tests',
    'significant_count_5pct',
    'significance_rate_5pct',
    'avg_coeff',
    'std_coeff'
]

df_5pct = df_5pct[column_order_5pct]

print(df_5pct)

              total_tests  significant_count_5pct  significance_rate_5pct  \
relationship                                                                
PM -> KAL             136                     125                0.919118   
CME -> KAL            136                     116                0.852941   
KAL -> PM             136                     114                0.838235   
CME -> PM             136                     106                0.779412   
KAL -> CME            136                     105                0.772059   
PM -> CME             136                      99                0.727941   

              avg_coeff  std_coeff  
relationship                        
PM -> KAL     -0.003852   0.007070  
CME -> KAL    -0.002062   0.003308  
KAL -> PM     -0.003367   0.003914  
CME -> PM     -0.002148   0.003091  
KAL -> CME    -0.002923   0.003522  
PM -> CME     -0.002797   0.003382  


### 2.3 Grouped by lag

In [7]:
# Function to calculate conditional stats (only for significant tests)
def get_conditional_stats(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats
    grouped = significant_data.groupby(['relationship',"lag_minutes"])[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# Process 5% Significance
df_5pct = VECM_master.groupby(['relationship',"lag_minutes"])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
df_5pct = df_5pct.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# Get avg coefficient and std for 5% significant results
stats_5pct = get_conditional_stats(VECM_master, 'is_sig_5pct', 'VECM_coefficient') # Replace 'granger_coeff' with your actual column name
df_5pct = df_5pct.join(stats_5pct)

# Sort and display
df_5pct = df_5pct.sort_values(by=["lag_minutes","significance_rate_5pct"], ascending = [True, False])

# Ordering the data in the desired manner before printing
column_order_5pct = [
    'total_tests',
    'significant_count_5pct',
    'significance_rate_5pct',
    'avg_coeff',
    'std_coeff'
]

df_5pct = df_5pct[column_order_5pct]

print(df_5pct)

                          total_tests  significant_count_5pct  \
relationship lag_minutes                                        
CME -> KAL   1                     34                      34   
KAL -> CME   1                     34                      34   
KAL -> PM    1                     34                      34   
PM -> CME    1                     34                      34   
PM -> KAL    1                     34                      34   
CME -> PM    1                     34                      33   
CME -> KAL   5                     34                      34   
KAL -> CME   5                     34                      34   
KAL -> PM    5                     34                      34   
PM -> CME    5                     34                      34   
PM -> KAL    5                     34                      34   
CME -> PM    5                     34                      32   
PM -> KAL    30                    34                      30   
KAL -> PM    30          

### 2.3 Differentiated at Volume

In [ ]:
volume_df = pd.read_csv("../../data/processed/Volume/volume.csv")
volume_df["tritile_CME"] =  pd.qcut(volume_df["CME"], q = 3, labels = ["low","medium","high"])
volume_df["tritile_PM"] = pd.qcut(volume_df["PM"], q =3, labels = ["low","medium","high"])
volume_df["tritile_Kalshi"] = pd.qcut(volume_df["Kalshi"],q=3, labels =["low", "medium","high"])

volume_info = volume_df[["event_date","contract", "tritile_CME","tritile_PM","tritile_Kalshi"]]
VECM_merged = pd.merge(VECM_master,volume_info, on = ["event_date","contract"], how = "left")

#### 2.3.1 CME

In [14]:
CME_vol_df = VECM_merged[VECM_merged["source_market"] == "CME"]

In [15]:
CME_vol_df

,event_date,contract,source_market,target_market,lag_minutes,VECM_coefficient,p_value,is_sig_5pct,is_raw_stationary,is_firstdiff_stationary,is_cointegrated,is_sig_1pct,relationship,tritile_CME,tritile_PM,tritile_Kalshi
0,2025-01-29,25bp_dec,CME,PM,1,-0.004708,0.000000,True,True,True,True,True,CME -> PM,high,medium,low
1,2025-01-29,25bp_dec,CME,PM,5,-0.002955,0.000000,True,True,True,True,True,CME -> PM,high,medium,low
2,2025-01-29,25bp_dec,CME,PM,30,-0.001106,0.002750,True,True,True,True,True,CME -> PM,high,medium,low
3,2025-01-29,25bp_dec,CME,PM,60,-0.000892,0.015714,True,True,True,True,False,CME -> PM,high,medium,low
8,2025-01-29,25bp_dec,CME,KAL,1,-0.002751,0.000000,True,True,True,True,True,CME -> KAL,high,medium,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,2026-04-29,0bp,CME,PM,60,-0.000142,0.009119,True,False,True,True,True,CME -> PM,medium,high,medium
800,2026-04-29,0bp,CME,KAL,1,-0.000453,0.000000,True,False,True,True,True,CME -> KAL,medium,high,medium
801,2026-04-29,0bp,CME,KAL,5,-0.000381,0.000000,True,False,True,True,True,CME -> KAL,medium,high,medium
802,2026-04-29,0bp,CME,KAL,30,-0.000249,0.000061,True,False,True,True,True,CME -> KAL,medium,high,medium


In [16]:
# Update the function to handle the new grouping level
def get_conditional_stats_tritle(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats by relationship AND tritle
    grouped = significant_data.groupby(['relationship', 'tritile_CME'])[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# Re-calculate the main summary (this matches your existing code)
CME_source_vol_95 = CME_vol_df.groupby(['relationship','tritile_CME'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
CME_source_vol_95 = CME_source_vol_95.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# Get avg coefficient and std for 5% significant results
# Make sure 'coef' is the correct column name in your DataFrame
stats_95 = get_conditional_stats_tritle(CME_vol_df, 'is_sig_5pct', 'VECM_coefficient')

# Join them together
CME_source_vol_95 = CME_source_vol_95.join(stats_95)

CME_source_vol_95 = CME_source_vol_95[["total_tests","significant_count_5pct","significance_rate_5pct","avg_coeff","std_coeff"]]

# Display the final result
print(CME_source_vol_95)

                          total_tests  significant_count_5pct  \
relationship tritile_CME                                        
CME -> KAL   low                   48                      44   
             medium                40                      32   
             high                  48                      40   
CME -> PM    low                   48                      43   
             medium                40                      27   
             high                  48                      36   

                          significance_rate_5pct  avg_coeff  std_coeff  
relationship tritile_CME                                                
CME -> KAL   low                        0.916667  -0.002126   0.002560  
             medium                     0.800000  -0.000758   0.001055  
             high                       0.833333  -0.003036   0.004668  
CME -> PM    low                        0.895833  -0.002594   0.003718  
             medium                     0

/tmp/ipykernel_748/222067266.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  CME_source_vol_95 = CME_vol_df.groupby(['relationship','tritile_CME'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
/tmp/ipykernel_748/222067266.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = significant_data.groupby(['relationship', 'tritile_CME'])[coeff_col].agg(['mean', 'std'])


#### 2.3.2 Polymarket

In [18]:
PM_vol_df = VECM_merged[VECM_merged["source_market"] == "PM"]

In [19]:
PM_vol_df

,event_date,contract,source_market,target_market,lag_minutes,VECM_coefficient,p_value,is_sig_5pct,is_raw_stationary,is_firstdiff_stationary,is_cointegrated,is_sig_1pct,relationship,tritile_CME,tritile_PM,tritile_Kalshi
4,2025-01-29,25bp_dec,PM,CME,1,-0.002570,0.000000,True,True,True,True,True,PM -> CME,high,medium,low
5,2025-01-29,25bp_dec,PM,CME,5,-0.001702,0.000000,True,True,True,True,True,PM -> CME,high,medium,low
6,2025-01-29,25bp_dec,PM,CME,30,-0.000948,0.002689,True,True,True,True,True,PM -> CME,high,medium,low
7,2025-01-29,25bp_dec,PM,CME,60,-0.000538,0.085975,False,True,True,True,False,PM -> CME,high,medium,low
16,2025-01-29,25bp_dec,PM,KAL,1,-0.004501,0.000000,True,True,True,True,True,PM -> KAL,high,medium,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799,2026-04-29,0bp,PM,CME,60,-0.000215,0.060426,False,False,True,True,False,PM -> CME,medium,high,medium
808,2026-04-29,0bp,PM,KAL,1,-0.002918,0.000000,True,False,True,True,True,PM -> KAL,medium,high,medium
809,2026-04-29,0bp,PM,KAL,5,-0.002410,0.000000,True,False,True,True,True,PM -> KAL,medium,high,medium
810,2026-04-29,0bp,PM,KAL,30,-0.001466,0.000000,True,False,True,True,True,PM -> KAL,medium,high,medium


In [20]:
# Update the function to handle the new grouping level
def get_conditional_stats_tritle(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats by relationship AND tritle
    grouped = significant_data.groupby(['relationship', 'tritile_PM'])[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# Re-calculate the main summary (this matches your existing code)
PM_source_vol_95 = PM_vol_df.groupby(['relationship','tritile_PM'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
PM_source_vol_95 = PM_source_vol_95.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# Get avg coefficient and std for 5% significant results
# Make sure 'coef' is the correct column name in your DataFrame
stats_95 = get_conditional_stats_tritle(PM_vol_df, 'is_sig_5pct', 'VECM_coefficient')

# Join them together
PM_source_vol_95 = PM_source_vol_95.join(stats_95)

PM_source_vol_95 = PM_source_vol_95[["total_tests","significant_count_5pct","significance_rate_5pct","avg_coeff","std_coeff"]]

# Display the final result
print(PM_source_vol_95)

                         total_tests  significant_count_5pct  \
relationship tritile_PM                                        
PM -> CME    low                  44                      36   
             medium               44                      32   
             high                 48                      31   
PM -> KAL    low                  44                      43   
             medium               44                      36   
             high                 48                      46   

                         significance_rate_5pct  avg_coeff  std_coeff  
relationship tritile_PM                                                
PM -> CME    low                       0.818182  -0.002702   0.002935  
             medium                    0.727273  -0.004515   0.004405  
             high                      0.645833  -0.001134   0.001116  
PM -> KAL    low                       0.977273  -0.001637   0.001737  
             medium                    0.818182  -0.008

/tmp/ipykernel_748/2042352927.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  PM_source_vol_95 = PM_vol_df.groupby(['relationship','tritile_PM'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
/tmp/ipykernel_748/2042352927.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = significant_data.groupby(['relationship', 'tritile_PM'])[coeff_col].agg(['mean', 'std'])


#### 2.3.3 Kalshi

In [21]:
Kalshi_vol_df = VECM_merged[VECM_merged["source_market"] == "KAL"]

In [22]:
# Update the function to handle the new grouping level
def get_conditional_stats_tritle(df, sig_col, coeff_col):
    # Filter to only keep significant results
    significant_data = df[df[sig_col] == 1]

    # Calculate group stats by relationship AND tritle
    grouped = significant_data.groupby(['relationship', 'tritile_Kalshi'])[coeff_col].agg(['mean', 'std'])
    return grouped.rename(columns={'mean': 'avg_coeff', 'std': 'std_coeff'})

# Re-calculate the main summary (this matches your existing code)
Kalshi_source_vol_95 = Kalshi_vol_df.groupby(['relationship','tritile_Kalshi'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
Kalshi_source_vol_95 = Kalshi_source_vol_95.rename(columns={
    'sum': 'significant_count_5pct',
    'count': 'total_tests',
    'mean': 'significance_rate_5pct'
})

# Get avg coefficient and std for 5% significant results
# Make sure 'coef' is the correct column name in your DataFrame
stats_95 = get_conditional_stats_tritle(Kalshi_vol_df, 'is_sig_5pct', 'VECM_coefficient')

# Join them together
Kalshi_source_vol_95 = Kalshi_source_vol_95.join(stats_95)

Kalshi_source_vol_95 = Kalshi_source_vol_95[["total_tests","significant_count_5pct","significance_rate_5pct","avg_coeff","std_coeff"]]

# Display the final result
print(Kalshi_source_vol_95)

                             total_tests  significant_count_5pct  \
relationship tritile_Kalshi                                        
KAL -> CME   low                      48                      40   
             medium                   40                      29   
             high                     48                      36   
KAL -> PM    low                      48                      41   
             medium                   40                      32   
             high                     48                      41   

                             significance_rate_5pct  avg_coeff  std_coeff  
relationship tritile_Kalshi                                                
KAL -> CME   low                           0.833333  -0.003165   0.003214  
             medium                        0.725000  -0.003452   0.004956  
             high                          0.750000  -0.002228   0.002217  
KAL -> PM    low                           0.854167  -0.001862   0.002350  

/tmp/ipykernel_748/1955408113.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  Kalshi_source_vol_95 = Kalshi_vol_df.groupby(['relationship','tritile_Kalshi'])['is_sig_5pct'].agg(['sum', 'count', 'mean'])
/tmp/ipykernel_748/1955408113.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = significant_data.groupby(['relationship', 'tritile_Kalshi'])[coeff_col].agg(['mean', 'std'])
